# 20 -- N=128 gamma_* family: reconstructed spectra across gamma_* = 2..20

For each `gamma_* = 2, 3, ..., 20` (19 values), build a fresh `N_b=128`
simultaneous-nucleation realization sized to that target `gamma_*`
(`box_size = R_*(gamma_*) * N_b^(1/3)`, following the same
`characteristic_scale`/`place_simultaneous_bubbles` convention already used
by `ptbuilder.weight_scan_diagnostics` and `08_chw_gamma_families.ipynb`),
compute its pairwise collision weights, and reconstruct its GW spectrum from
the `gs1-20` production scan (`19_production_scan.ipynb`) -- the only scan
with enough `gamma_ij`/`kR_*` coverage for pair gammas up to
`pair_gamma_factor(2.5) * 20 = 50`.

`generate_surrogate` sizes every row's `omega_max` as `KR_MAX /
rstar_at_gamma(gamma_ij / pair_gamma_factor)` -- i.e. each row's raw `wlist`
was deliberately built so that, converted through the natural `R_*` of the
smallest target realization that would actually produce a pair at that
`gamma_ij`, it reaches exactly `kR_*=KR_MAX` at real computed data (and
`omega_min` carries headroom below `KR_MIN`). So for a `gamma_*` case here,
reconstructing directly onto a `k` grid spaced as
`geomspace(KR_MIN, KR_MAX, N_PLOT) / R_*(gamma_*)` lands exactly on that
built-in coverage -- no interpolation-driven extrapolation, no masking,
every curve spans `kR_* in [KR_MIN, KR_MAX]` exactly. (An earlier version of
this notebook projected every case onto one shared grid borrowed from a
single scan row and then masked the result -- that's what produced curves
that didn't quite reach the endpoints; this version queries the scan the way
`KR_MIN`/`KR_MAX` were designed for.)

One realization per `gamma_*` (no ensemble averaging), weights run locally
(`N_b=128` is small enough not to need SLURM). All 19 reconstructed spectra
are plotted together in CHW's dimensionless units, colored by `gamma_*`.

In [ ]:
from pathlib import Path

import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from scipy.interpolate import interp1d

import ptbuilder as pt
from ptbuilder.analysis import load_scan, load_weights
from ptbuilder.weight_scan_diagnostics import run_grid

ROOT = Path(pt.__file__).parent.parent
DATA = ROOT / 'data'

LAMBDA_BAR = 0.84
GAMMA_STARS = np.arange(2, 21)   # 2, 3, ..., 20
N_B = 128
N_T = 1024                       # weights time resolution per case
MAX_T_OVER_RSTAR = 2.5           # weights time window, in units of that case's own R_*
BASE_SEED = 128020

NEW_SCAN_DIR = DATA / 'runtime_scan_lb0.84_gs1-20_kR1-100_nw32'
NEW_SCAN_KR_MIN, NEW_SCAN_KR_MAX = 1.0, 100.0   # KR_MIN/KR_MAX from 19_production_scan.ipynb

N_PLOT = 200   # points per reconstructed curve
X_PLOT = np.geomspace(NEW_SCAN_KR_MIN, NEW_SCAN_KR_MAX, N_PLOT)   # shared kR_* axis -- same for every case

lb_tag = f'{LAMBDA_BAR:g}'.replace('.', 'p')
CASE_ROOT = DATA / f'gamma_star_weights_lb{lb_tag}_N{N_B}'   # matches run_grid's own naming

## 1. Build the 19 N=128 realizations and run weights locally

Reuses `ptbuilder.weight_scan_diagnostics.run_grid` as-is: for each
`gamma_*` it solves for `R_*(gamma_*)` from the instanton kinematics, sizes
the box as `R_* * N_b^(1/3)`, places `N_b` bubbles (periodic random
sequential placement with a hard midpoint-separation exclusion), writes
`weights_input.h5`, and runs `cpp/weights/weights` on it -- everything lands
under `data/gamma_star_weights_lb0p84_N128/gamma_star_{NNN}/`. `workers` x
`omp_threads` is kept modest (8 total) to match this machine, not a cluster
node.

In [ ]:
_ = run_grid(
    ROOT, gamma_stars=GAMMA_STARS, n_b=N_B, n_t=N_T,
    max_t_over_rstar=MAX_T_OVER_RSTAR, base_seed=BASE_SEED,
    workers=4, omp_threads=2, lambda_bar=LAMBDA_BAR,
)

## 2. Load the gs1-20 production scan

Self-contained, non-rectangular scan -- each `gamma_ij` row has its own
`times`/`wlist`. `build_scan_interp(k_out)` projects every row onto whatever
output `k` grid it's given and returns a `(gamma_ij, t) -> spectrum`
interpolator for that grid (log-linear across each row's own `times`,
log-linear across the two bracketing `gamma_ij` rows) -- parameterized
rather than built once, because (per the intro) the right output grid is
per-`gamma_*`, not one grid shared across every case.

In [ ]:
scan = load_scan(NEW_SCAN_DIR)
scan_gammas = np.array([r.gamma_ij for r in scan.rows])
scan_t_bounds = np.array([min(r.times[0] for r in scan.rows), scan.t_max])


def build_scan_interp(k_out):
    """(gamma_ij, t) -> spectrum interpolator projected onto k_out."""
    Nk = len(k_out)
    spec_on_kout = []
    for r in scan.rows:
        spec_k = np.zeros((len(r.times), Nk))
        for it in range(len(r.times)):
            spec_k[it] = np.interp(k_out, r.wlist, r.spectrum[it], left=0., right=0.)
        spec_on_kout.append(spec_k)

    floor = max(s.max() for s in spec_on_kout) * 1e-12
    log_floor = np.log(floor)

    row_log_interp = []
    for r, spec_k in zip(scan.rows, spec_on_kout):
        log_spec = np.log(np.maximum(spec_k, floor))
        row_log_interp.append(interp1d(
            r.times, log_spec, axis=0, kind='linear', bounds_error=False,
            fill_value=(log_floor, log_spec[-1]),
        ))

    def interp_fn(pts):
        g_arr, t_arr = pts[:, 0], pts[:, 1]
        out = np.empty((len(pts), Nk))
        for g_val in np.unique(g_arr):
            sel = g_arr == g_val
            g_clip = float(np.clip(g_val, scan_gammas.min(), scan_gammas.max()))
            ig_hi = int(np.searchsorted(scan_gammas, g_clip))
            ig_hi = min(max(ig_hi, 1), len(scan_gammas) - 1)
            ig_lo = ig_hi - 1
            g_lo, g_hi = scan_gammas[ig_lo], scan_gammas[ig_hi]
            frac = 0.0 if g_hi == g_lo else (g_clip - g_lo) / (g_hi - g_lo)
            log_lo = row_log_interp[ig_lo](t_arr[sel])
            log_hi = row_log_interp[ig_hi](t_arr[sel])
            out[sel] = np.exp(log_lo + frac * (log_hi - log_lo))
        return out

    return interp_fn

## 3. Reconstruct each realization's spectrum

`bm_reconstruct` steps at the fine weights-time resolution (not the coarse
scan grid), matching `06_n64_reconstruction.ipynb`. Each case's own
`R_star`/`L` (written into its `weights_input.h5` by `prepare_case`) are
read back rather than recomputed. Reconstruction happens directly on
`k_plot = X_PLOT / R_star` -- the same `R_*` that sized this realization's
box -- so the spectrum comes back already sampled at exactly
`kR_* = X_PLOT` (see the intro for why this lines up with real, non-
extrapolated scan data at both ends).

In [ ]:
def bm_reconstruct(gamma, weights_t, weights_times, t_max, interp_fn, times, gammas):
    g = float(np.clip(gamma, gammas.min(), gammas.max()))
    t_hi = min(t_max, times[-1])
    mask = (weights_times >= times[0]) & (weights_times <= t_hi)
    t_comp = weights_times[mask]
    if len(t_comp) == 0 or t_comp[-1] < t_hi:
        t_comp = np.append(t_comp, t_hi)
    w = np.interp(t_comp, weights_times, weights_t)
    pts = np.column_stack([np.full(len(t_comp), g), t_comp])
    specs = interp_fn(pts)
    delta = np.diff(specs, axis=0)
    return (w[:len(t_comp) - 1, None] * delta).sum(axis=0)


reconstructions = {}
case_meta = {}
for gamma_star in GAMMA_STARS:
    case_dir = CASE_ROOT / f'gamma_star_{int(gamma_star):03d}'
    inp = case_dir / 'weights_input.h5'
    out = case_dir / 'weights_output.h5'
    with h5py.File(inp, 'r') as f:
        R_star = float(f.attrs['R_star'])
        L_box = float(f.attrs['L'])
        weights_times = f['t'][:]

    weights, pair_i, pair_j, pair_gammas = load_weights(out, N_T)

    k_plot = X_PLOT / R_star   # exactly kR_* = X_PLOT once multiplied back by R_star
    interp_fn = build_scan_interp(k_plot)

    spectrum = np.zeros(N_PLOT)
    for w_row, g in zip(weights, pair_gammas):
        spectrum += bm_reconstruct(float(g), w_row, weights_times, weights_times[-1],
                                   interp_fn, scan_t_bounds, scan_gammas)

    reconstructions[float(gamma_star)] = spectrum
    case_meta[float(gamma_star)] = dict(R_star=R_star, L_box=L_box, n_pairs=len(pair_i))
    print(f'gamma_*={gamma_star:2d}: R_*={R_star:.4f}, L={L_box:.2f}, '
          f'{len(pair_i)} contributing pairs, gamma_ij in '
          f'[{pair_gammas.min():.3f}, {pair_gammas.max():.3f}]')

## 4. Convert to CHW units and plot the gamma_* family

Each curve's spectrum was reconstructed directly on `k_plot = X_PLOT /
R_star` (its own realization's `R_*`), so `energy_to_chw`'s `x` output is
`X_PLOT` again for every case, exactly -- no masking needed. Curves are
colored by `gamma_*` with a colorbar, matching the style already used in
`ptbuilder.weight_scan_diagnostics.plot_diagnostics`.

In [ ]:
def rho_vac_bar(lambda_bar):
    """False-minus-true vacuum energy in BM dimensionless units."""
    U = 3.0 + np.sqrt(9.0 - 8.0 * lambda_bar)
    V_true = 0.5 - U / (4.0 * lambda_bar) + U**2 / (32.0 * lambda_bar)
    return -V_true


def energy_to_chw(k, dE_dlnk, Rstar, lambda_bar, volume):
    rho_vac = rho_vac_bar(lambda_bar)
    x = np.asarray(k) * Rstar
    y = (3.0 / (8.0 * np.pi * rho_vac**2 * Rstar**2 * volume)) * np.asarray(dE_dlnk)
    return x, y


norm = Normalize(vmin=GAMMA_STARS.min(), vmax=GAMMA_STARS.max())
cmap = plt.get_cmap('viridis')

fig, ax = plt.subplots(figsize=(8.5, 5.5))
for gamma_star in GAMMA_STARS:
    meta = case_meta[float(gamma_star)]
    k_plot = X_PLOT / meta['R_star']
    x, y = energy_to_chw(k_plot, reconstructions[float(gamma_star)],
                         meta['R_star'], LAMBDA_BAR, meta['L_box']**3)
    ax.plot(x, y, color=cmap(norm(gamma_star)), lw=1.6)

colorbar = fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), ax=ax)
colorbar.set_label(r'$\gamma_*$')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(r'$kR_*$')
ax.set_ylabel(r'$[H_*R_*\Omega_{\rm vac}]^{-2}\,d\Omega_{\rm gw}/d\ln k$')
ax.set_title(rf'$N_b={N_B}$ reconstructed spectra, $\gamma_*={GAMMA_STARS[0]}..{GAMMA_STARS[-1]}$, '
            rf'$\bar{{\lambda}}={LAMBDA_BAR}$')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(ROOT / 'n128_gamma_star_family.pdf', bbox_inches='tight')
plt.show()